# LAB: Un Agent HITL (Human-In-The-Loop)
## Human-In-The-Loop Agents for Production Systems
### Master SDIA - Prof. RETAL SARA

Ce notebook couvre la création d'agents avec intervention humaine explicite pendant l'exécution.

## 📚 Qu'est-ce qu'un agent HITL?

### **HITL = Human-In-The-Loop**

Un agent HITL est un agent d'IA dont l'exécution inclut **explicitement une intervention humaine** pendant le processus.

### Différent de:
- ❌ **Before Loop**: Instruction humaine avant l'exécution (prompt)
- ❌ **After Loop**: Validation humaine après l'exécution (vérification finale)
- ✅ **In The Loop**: Intervention humaine **PENDANT** l'exécution (approche moderne)

### Flux d'un agent HITL:

```
Agent exécute une tâche
    ↓
S'arrête à un point critique
    ↓
Demande l'approbation humaine
    ↓
Humain décide: Approuver / Rejeter / Modifier
    ↓
Agent continue avec la décision
    ↓
Résultat final
```

### Cas d'usage:
- 💰 **Finance**: Approbation avant virement important
- 📧 **Email**: Vérification avant envoi d'email
- 🏥 **Santé**: Validation avant traitement critique
- 📝 **Contrats**: Vérification avant signature
- 🛒 **Commandes**: Approbation avant traitement
- 🔐 **Sécurité**: Validation avant accès sensible

### Avantages:
- ✅ Contrôle et transparence
- ✅ Prévention des erreurs critiques
- ✅ Audit trail (historique)
- ✅ Responsabilité humaine
- ✅ Confiance en l'IA
- ✅ Conformité réglementaire

## ⚙️ SETUP - Installation et Configuration

In [ ]:
print("""
🚀 INSTALLATION DES DÉPENDANCES HITL
="*70

Exécutez dans votre terminal:

  uv add langchain langchain-ollama langchain-core \\
          langgraph python-dotenv

Packages clés:
  ✓ langchain - Framework principal
  ✓ langgraph - Middleware et interrupts
  ✓ langchain.agents.middleware - HumanInTheLoopMiddleware

Important:
  • HumanInTheLoopMiddleware est disponible dans langgraph
  • Gère les interruptions et reprises
  • Supporte approve/reject/edit
""")

---

## 🔧 Concept: Middleware

### Qu'est-ce qu'un Middleware?

In [ ]:
print("""
🔧 MIDDLEWARE - Le Cœur du HITL
="*70

Un middleware est un composant logiciel **intermédiaire** qui s'insère
entre deux étapes d'un système pour:

  ✓ Intercepter les données ou l'exécution
  ✓ Modifier ou valider avant passage
  ✓ Contrôler le flux d'exécution
  ✓ Enrichir avec des données supplémentaires
  ✓ Appliquer des règles ou politiques

Architecture avec middleware:

    Agent
       ↓
    Middleware
       ├─ Intercept
       ├─ Validate
       ├─ Wait for human
       ├─ Resume
       └─ Continue
       ↓
    Tool Execution
       ↓
    Result

 HumanInTheLoopMiddleware fournit:

  1️⃣ Interrupt (Arrêt):
     • S'arrête AVANT l'exécution d'outils spécifiques
     • Décision: {tool_name: True/False}

  2️⃣ Notification:
     • Informe l'humain de ce qui va se passer
     • Affiche les arguments du tool

  3️⃣ Attente:
     • Pause l'exécution
     • Attend la décision humaine

  4️⃣ Reprise:
     • Resume avec la décision
     • Peut approuver, rejeter ou modifier
""")

---

## 📧 PARTIE 1: Définition des outils

In [ ]:
from langchain.tools import tool, ToolRuntime

print("\n📧 OUTILS POUR UN AGENT EMAIL\n")
print("="*70)

# Outil 1: Lire un email
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the state."""
    try:
        email_content = runtime.state["email"]
        return f"Email content: {email_content}"
    except KeyError:
        return "No email found in state"

print("✅ Tool 1: read_email")
print("   Description: Read an email from the state")
print("   Access: runtime.state['email']")
print()

# Outil 2: Envoyer un email
@tool
def send_email(body: str) -> str:
    """Send an email with the given body."""
    # En production, ceci enverrait un vrai email
    return f"✓ Email sent with body: {body[:50]}..."

print("✅ Tool 2: send_email")
print("   Description: Send an email with the given body")
print("   Parameters: body (str) - Le contenu de l'email")
print()

# Outil 3: Créer un brouillon
@tool
def draft_response(email_text: str, response_draft: str) -> str:
    """Create a draft response to an email."""
    return f"Draft created:\n{response_draft}"

print("✅ Tool 3: draft_response")
print("   Description: Create a draft response")
print("   Parameters: email_text, response_draft")
print()

print("="*70)
print("Ces outils seront utilisés dans un agent HITL")

---

## 🤖 PARTIE 2: Création d'un agent HITL

In [ ]:
print("""
🤖 AGENT HITL AVEC MIDDLEWARE
="*70

Code pour créer un agent avec HumanInTheLoopMiddleware:

```python
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt
from langchain_ollama import ChatOllama

# 1. Définir l'état
class EmailState(AgentState):
    email: str  # Contenu de l'email à traiter

# 2. Créer l'agent
model = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

agent = create_agent(
    model=model,
    tools=[read_email, send_email, draft_response],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    system_prompt=(
        "Tu es un assistant email. "
        "Lis les emails et crée une réponse appropriée. "
        "Assure-toi que la réponse est professionnelle et courtoise."
    )
)
```

Avec middleware HITL:

```python
# Option 1: Interruption simple
agent = create_agent(
    model=model,
    tools=[read_email, send_email, draft_response],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    # Interrupts AVANT send_email
    interrupt_before=["send_email"]
)

# Option 2: Middleware avancé
from langgraph.types import interrupt

agent.add_interrupt("send_email")
```


Configuration des interruptions:

  interrupt_before=["send_email"]
    └─ S'arrête AVANT l'exécution de send_email
    └─ Permet à l'humain d'approuver/modifier/rejeter

  interrupt_after=["read_email"]
    └─ S'arrête APRÈS l'exécution de read_email
    └─ Pour vérifier ce qui a été lu
""")

print("\n✨ Points clés:")
print("  • create_agent() accepte interrupt_before/after")
print("  • InMemorySaver conserve l'état entre requêtes")
print("  • thread_id pour identifier la session")

### Exemple complet d'agent HITL

In [ ]:
print("""
💻 EXEMPLE COMPLET: Agent Email HITL

```python
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

# Définir l'état
class EmailState(AgentState):
    email: str = ""

# Créer l'agent avec interruption
model = ChatOllama(model="llama3.2:3b", temperature=0)

agent = create_agent(
    model=model,
    tools=[read_email, send_email, draft_response],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    interrupt_before=["send_email"],  # ← HITL ici!
    system_prompt="Tu es un assistant email intelligent..."
)

# Configuration de session
config = {"configurable": {"thread_id": "user123"}}

# Première invocation: Agent lit et crée un brouillon
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Lis mon email et prépare une réponse."
            )
        ],
        "email": "Bonjour, je serai en retard demain. Pouvons-nous reprogrammer?"
    },
    config=config
)

print(response)
# → Agent s'arrête AVANT send_email
# → Attend la décision humaine
```
""")

---

## ✅ PARTIE 3: Approuver le résultat

In [ ]:
print("""
✅ APPROUVER L'ACTION
="*70

Code pour approuver et continuer:

```python
from langgraph.types import Command

# L'agent s'était arrêté en attente d'approbation
# Maintenant on l'approuve et on continue

response = agent.invoke(
    Command(resume=True),  # Reprendre avec approbation
    config=config  # Même thread_id
)

print(response['messages'][-1].content)
# → "Email sent successfully!"
```

Flux d'exécution avec approbation:

    1. Agent détecte: "Je dois envoyer un email"
       ↓
    2. Middleware intercepte (interrupt_before["send_email"])
       ↓
    3. S'arrête et attend décision humaine
       ↓
    4. Humain: "OK, envoie l'email" (resume=True)
       ↓
    5. Agent continue avec send_email()
       ↓
    6. Email envoyé
       ↓
    7. Exécution terminée

Cas d'usage Approve:
    ✓ Email est bon, on envoie
    ✓ Transfert d'argent est correct, on valide
    ✓ Traitement médical approuvé
    ✓ Contrat signé, on procède
""")

---

## ❌ PARTIE 4: Refuser le résultat

In [ ]:
print("""
❌ REJETER L'ACTION
="*70

Code pour rejeter et arrêter:

```python
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume=False,  # Rejeter
        message="Cette réponse n'est pas appropriée. Demande de réécrire."
    ),
    config=config  # Même thread_id
)

print(response)
# → Agent reçoit le rejet et peut réessayer
```

Flux d'exécution avec rejet:

    1. Agent s'arrête en attente
       ↓
    2. Humain: "NON, n'envoie pas ça!"
       ↓
    3. Agent reçoit le rejet + message
       ↓
    4. Agent peut:
       ├─ Réécrire l'email
       ├─ Changer la stratégie
       └─ Demander clarification
       ↓
    5. Nouvelle tentative ou arrêt

Cas d'usage Reject:
    ✗ Email contient une erreur
    ✗ Montant de transfert incorrect
    ✗ Diagnostic médical douteux
    ✗ Contrat contient des clauses problématiques
    ✗ Recommandation non pertinente
""")

print("\n💡 Le rejet n'arrête pas toujours l'agent:")
print("    L'agent peut être programmé pour réessayer")
print("    Ou demander à l'humain de préciser")

---

## ✏️ PARTIE 5: Modifier le résultat

In [ ]:
print("""
✏️ MODIFIER L'ACTION
="*70

Code pour modifier et continuer:

```python
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume=True,
        edited_action={
            "name": "send_email",
            "args": {
                "body": "Bonjour, je dois malheureusement annuler "
                        "notre réunion. Je vous présente mes excuses. "
                        "Pourriez-nous programmer à la semaine prochaine?"
            }
        }
    ),
    config=config  # Même thread_id
)

print(response['messages'][-1].content)
# → Email envoyé avec le NOUVEAU contenu
```

Flux d'exécution avec modification:

    1. Agent propose: "Envoyer email avec texte X"
       ↓
    2. Middleware arrête en attente
       ↓
    3. Humain: "OK mais change le texte en Y"
       └─ Envoie edited_action avec nouveau texte
       ↓
    4. Agent exécute avec le texte MODIFIÉ
       ↓
    5. Email envoyé avec le nouveau contenu
       ↓
    6. Exécution terminée

Cas d'usage Edit:
    ✏️ Email bon mais un peu sec, ajouter courtoisie
    ✏️ Transfert correct mais mauvais bénéficiaire
    ✏️ Traitement approuvé mais dose différente
    ✏️ Contrat prêt mais clauses à ajuster
    ✏️ Recommandation bonne mais changer le timing

Avantage Edit vs Reject:
    • Plus efficace: Pas besoin de refaire tout le processus
    • Contrôle fin: L'humain peut ajuster précisément
    • Apprentissage: L'agent voir comment on corrige
""")

---

## 📊 Résumé des 3 modes de décision

In [ ]:
print("""
📊 LES 3 DÉCISIONS POSSIBLES
="*70

┌─────────────────┬──────────────────┬──────────────────┐
│ Décision        │ Code             │ Résultat         │
├─────────────────┼──────────────────┼──────────────────┤
│ APPROVE         │ resume=True      │ Execute as-is    │
│                 │                  │ Action approuvée │
│                 │                  │ Cas: ✓ OK, go!   │
├─────────────────┼──────────────────┼──────────────────┤
│ REJECT          │ resume=False +   │ Action rejetée   │
│                 │ message          │ Agent réessaie   │
│                 │                  │ Cas: ✗ Bad idea  │
├─────────────────┼──────────────────┼──────────────────┤
│ EDIT            │ edited_action +  │ Execute modifié  │
│                 │ args             │ Paramètres changés
│                 │                  │ Cas: ~ Almost!   │
└─────────────────┴──────────────────┴──────────────────┘

Exemple d'utilisation:

    # Approve
    agent.invoke(Command(resume=True), config)
    
    # Reject
    agent.invoke(Command(resume=False, message="Why not"), config)
    
    # Edit
    agent.invoke(Command(
        resume=True,
        edited_action={
            "name": "send_email",
            "args": {"body": "Nouveau texte"}
        }
    ), config)

Production patterns:

    Finances:
        Approve: ✓ Transfert autorisé
        Reject: ✗ Montant suspect
        Edit: ~ Correct le bénéficiaire
    
    Email:
        Approve: ✓ Message professionnel
        Reject: ✗ Contient erreurs
        Edit: ~ Améliorer la formulation
    
    Santé:
        Approve: ✓ Traitement approprié
        Reject: ✗ Allergie non considérée
        Edit: ~ Ajuster la dose
""")

---

## 🏗️ Architecture complète HITL

In [ ]:
print("""
🏗️ ARCHITECTURE COMPLETE D'UN SYSTÈME HITL
="*70

Front-end UI:
    ┌─────────────────────────────────────┐
    │     Interface Utilisateur           │
    │  [Approve] [Reject] [Edit] buttons  │
    │  Affiche: Proposition de l'agent    │
    └────────────┬────────────────────────┘
                 │
                 ↓
Human Decision Layer:
    ┌─────────────────────────────────────┐
    │  Serveur de décision humaine        │
    │  • Validate user input              │
    │  • Create Command object            │
    │  • Send to agent                    │
    └────────────┬────────────────────────┘
                 │
                 ↓
Agent Layer:
    ┌─────────────────────────────────────┐
    │     Agent HITL                      │
    │  • Execute tools                    │
    │  • Middleware intercepts            │
    │  • Waits for human decision         │
    │  • Resumes with Command             │
    │  • Saves state via checkpointer     │
    └────────────┬────────────────────────┘
                 │
                 ↓
Tool Layer:
    ┌─────────────────────────────────────┐
    │     Tools (Approved/Modified)       │
    │  • send_email                       │
    │  • transfer_money                   │
    │  • update_database                  │
    │  • etc.                             │
    └────────────┬────────────────────────┘
                 │
                 ↓
External Systems:
    ┌─────────────────────────────────────┐
    │     Email, Bank, DB, APIs           │
    └─────────────────────────────────────┘

Data Flow with State:
    
    Initial State:
    {
        "messages": [...],
        "email": "Email content",
        "user_id": "123"
    }
        ↓ (Agent processes)
    Interruption Point:
    Agent stopped, awaiting decision
        ↓ (Human decides)
    Human Input:
    Command(resume=True, edited_action={...})
        ↓ (Agent continues)
    Final State:
    {
        "messages": [..., tool_result, human_decision],
        "email": "...",
        "user_id": "123"
    }
        ↓ (Saved via checkpointer)
    Persistence:
    State available for next request with same thread_id
""")

---

## 💼 Cas d'usage réels

In [ ]:
print("""
💼 CAS D'USAGE RÉELS HITL
="*70

1️⃣ AGENT BANCAIRE
   Tâche: Traiter une demande de transfert
   HITL: Approuver avant envoi (montant, bénéficiaire)
   Interruptions:
     • interrupt_before=["transfer_money"]
     • interrupt_before=["approve_transaction"]
   Décisions possibles:
     ✓ Approver: Transfert autorisé
     ✗ Rejeter: Montant suspect
     ~ Éditer: Corriger le bénéficiaire

2️⃣ ASSISTANT EMAIL PROFESSIONNEL
   Tâche: Répondre aux emails importants
   HITL: Vérifier avant envoi (ton, contenu)
   Interruptions:
     • interrupt_before=["send_email"]
     • interrupt_before=["send_important_email"]
   Décisions:
     ✓ Approver: Email professionnel
     ✗ Rejeter: Trop informel ou contient erreur
     ~ Éditer: Améliorer la formulation

3️⃣ SYSTÈME DE DIAGNOSTIC MÉDICAL
   Tâche: Analyser symptômes et proposer traitement
   HITL: Médecin approuve avant prescription
   Interruptions:
     • interrupt_before=["prescribe_medication"]
     • interrupt_before=["schedule_surgery"]
   Décisions:
     ✓ Approver: Traitement approprié
     ✗ Rejeter: Allergie non considérée
     ~ Éditer: Ajuster dose ou alternative

4️⃣ AGENT DE RESSOURCES HUMAINES
   Tâche: Traiter demande de congé
   HITL: Gestionnaire approuve avant confirmation
   Interruptions:
     • interrupt_before=["approve_leave"]
     • interrupt_before=["update_payroll"]
   Décisions:
     ✓ Approver: Congé approuvé
     ✗ Rejeter: Période insuffisante avant projet
     ~ Éditer: Ajuster dates

5️⃣ AGENT DE CONFORMITÉ
   Tâche: Auditer contrats
   HITL: Avocat approuve avant signature
   Interruptions:
     • interrupt_before=["sign_contract"]
     • interrupt_before=["approve_terms"]
   Décisions:
     ✓ Approver: Termes acceptables
     ✗ Rejeter: Clause problématique
     ~ Éditer: Ajouter clause protective

6️⃣ AGENT E-COMMERCE
   Tâche: Valider commande
   HITL: Vérifier avant traitement (stock, fraude)
   Interruptions:
     • interrupt_before=["process_payment"]
     • interrupt_before=["ship_order"]
   Décisions:
     ✓ Approver: Commande valide
     ✗ Rejeter: Potentielle fraude
     ~ Éditer: Changer adresse livraison

Avantages communs:
    ✅ Prévention d'erreurs critiques
    ✅ Audit trail complet
    ✅ Responsabilité humaine
    ✅ Confiance dans le système
    ✅ Conformité réglementaire
    ✅ Amélioration continue via feedback
""")

---

## 🎯 Bonnes pratiques HITL

In [ ]:
print("""
✨ BONNES PRATIQUES HITL
="*70

1️⃣ Points d'interruption stratégiques
   ✓ Placer les interruptions AVANT les actions critiques
   ✓ Ne pas interrompre trop souvent (fatigue décisionnelle)
   ✓ Adapter au risque: Haut risque = plus d'interruptions
   ✗ Éviter: Interrompre chaque étape

2️⃣ Information claire
   ✓ Expliquer pourquoi l'agent veut faire l'action
   ✓ Afficher les paramètres détaillés
   ✓ Montrer les conséquences possibles
   ✓ Fournir contexte et historique
   ✗ Éviter: Information minimaliste

3️⃣ Feedback humain
   ✓ Enregistrer chaque décision humaine
   ✓ Analyser les patterns de rejet
   ✓ Utiliser pour améliorer l'agent
   ✓ Audit trail complet
   ✗ Éviter: Ignorer le feedback

4️⃣ Performance
   ✓ Caching des décisions répétitives
   ✓ Timeout raisonnables (5-30 min)
   ✓ Alertes sur interruptions trop longues
   ✓ Notifications push/email si besoin
   ✗ Éviter: Laisser l'agent bloqué indéfiniment

5️⃣ Escalade
   ✓ Si rejet répété: Escalader au manager
   ✓ Définir les seuils de rejet
   ✓ Notifications pour décisions importantes
   ✓ Audit trail pour compliance
   ✗ Éviter: Ignorer les refus répétés

6️⃣ Sécurité
   ✓ Authentifier l'utilisateur qui décide
   ✓ Vérifier les permissions
   ✓ Chiffrer les données sensibles
   ✓ Logging de toutes les modifications
   ✗ Éviter: Pas de contrôle d'accès

7️⃣ UX Design
   ✓ Interfaceclaire et intuitive
   ✓ Boutons obvious: [Approve] [Reject] [Edit]
   ✓ Afficher les options en clair
   ✓ Shortcats clavier pour rapidité
   ✓ Mobile-friendly si possible
   ✗ Éviter: Interface confuse

8️⃣ Documentation
   ✓ Documenter chaque point d'interruption
   ✓ Expliquer les critères de rejet
   ✓ Fournir guide de l'utilisateur
   ✓ Exemples de décisions bonnes/mauvaises
   ✗ Éviter: Pas de doc
""")

---

## 📝 Résumé

In [ ]:
print("""
📝 RÉSUMÉ - AGENT HITL EN 30 SECONDES
="*70

✅ HITL = Human-In-The-Loop
   → Agent s'arrête PENDANT l'exécution
   → Demande approbation humaine
   → Continue avec la décision

✅ LES 3 DÉCISIONS
   1. APPROVE (resume=True)
      → Agent exécute comme prévu
   2. REJECT (resume=False)
      → Action rejetée, agent réessaie
   3. EDIT (edited_action={})
      → Agent exécute avec paramètres modifiés

✅ MIDDLEWARE
   → Composant intermédiaire
   → Intercepte avant/après actions
   → Gère les interruptions
   → Permet le contrôle humain

✅ ARCHITECTURE
   Agent
     ↓
   [Middleware] ← Interruption
     ↓
   ⏸️ Wait for Human
     ↓
   Human Decision (Approve/Reject/Edit)
     ↓
   Resume with Command
     ↓
   Tool Execution
     ↓
   State Saved

✅ POINTS D'INTERRUPTION
   interrupt_before=["tool_name"]
     → S'arrête AVANT l'exécution
   interrupt_after=["tool_name"]
     → S'arrête APRÈS l'exécution

✅ STATE MANAGEMENT
   • InMemorySaver pour sauvegarder l'état
   • thread_id pour identifier la session
   • Historique complet préservé
   • Reprendre depuis l'interruption

✅ PRODUCTION USE CASES
   • Transferts bancaires (Approve avant envoi)
   • Emails importants (Vérifier avant envoyer)
   • Prescriptions médicales (Médecin approuve)
   • Contrats (Avocat valide)
   • Ressources humaines (Manager approuve)
   • Conformité (Audit trail complet)

✨ AVANTAGE CLÉS
   ✓ Prévention des erreurs
   ✓ Audit trail complet
   ✓ Responsabilité humaine
   ✓ Confiance dans l'IA
   ✓ Conformité réglementaire
   ✓ Amélioration continue

🚀 C'EST VOTRE DERNIER LAB!
   Vous maîtrisez maintenant:
   ✅ Prompt Engineering
   ✅ Agents LangChain
   ✅ State & Context
   ✅ RAG & SQL
   ✅ MCP Protocol
   ✅ LangGraph Studio
   ✅ HITL Agents (CE LAB!)
   
   → Prêt pour construire des agents en production! 🎉
""")

print("\n" + "="*70)
print("✨ Félicitations! Vous avez terminé l'intégralité du curriculum!")
print("="*70)